# 10. Feature Selection

This notebook performs feature selection on the encoded and pre-filtered dataset to identify the most relevant features for predicting student dropout.

## Objectives
- Load encoded and pre-filtered data from previous steps
- Explore dataset characteristics and structure
- Perform feature selection techniques to reduce dimensionality
- Evaluate feature importance and select optimal subset
- Prepare final feature set for model training

## Table of Contents
1. [Setup & Data Loading](#setup--data-loading)
2. [Data Exploration](#data-exploration)
3. [Feature Selection Methods](#feature-selection-methods)
   - 3.1 [Variance Threshold Filtering](#variance-threshold-filtering)
   - 3.2 [Correlation Analysis](#correlation-analysis)
   - 3.3 [Permutation Importance](#permutation-importance)
4. [Feature Evaluation](#feature-evaluation)
5. [Final Feature Set](#final-feature-set)

---

## 1. Setup & Data Loading {#setup--data-loading}

Import required libraries and load the encoded dataset from previous preprocessing steps.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import RepeatedStratifiedKFold

In [ ]:
# Define paths
current_user = os.getlogin()
PROCESSED_DIR = Path(f"/home/{current_user}/local-share/processed")
OUT_DIR = PROCESSED_DIR

# Load data
data = pd.read_csv(PROCESSED_DIR / "encoded_and_split_data.csv", sep=None, engine="python", encoding="utf-8-sig")
print(f"Data shape: {data.shape}")
data.head()

In [ ]:
data['sdo1_previous_Multiple_Previous_Education'].value_counts(dropna=False)

## 2. Data Exploration {#data-exploration}

Examine the structure and characteristics of the loaded dataset before applying feature selection.

In [ ]:
# Basic data exploration
print(f"Dataset shape: {data.shape}")
print(f"Columns: {len(data.columns)}")
print(f"Memory usage: {data.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

# Check data types
print(f"\nData types distribution:")
print(data.dtypes.value_counts())

# Check for missing values
missing_summary = data.isnull().sum()
missing_cols = missing_summary[missing_summary > 0]
if len(missing_cols) > 0:
    print(f"\nColumns with missing values: {len(missing_cols)}")
    print(missing_cols.head(10))
else:
    print("\nNo missing values found")

# Check target distribution if available
if 'is_dropout' in data.columns:
    print(f"\nTarget distribution:")
    print(data['is_dropout'].value_counts())

## 3. Feature Selection Methods {#feature-selection-methods}

Apply feature selection techniques to identify the most important features for dropout prediction:
- variance threshold filtering
- feature correlation
- permutation importance

Feature selection is based on the training dataset only.

In [ ]:
# Columns to drop based on feature selection
columns_to_drop = [
    # Add columns to drop here after feature selection
    'attended_online_opleidingspresentatie',                        # Feature does not have a value in training set
    'sdo2_orientation_First_Event_Date_missing_flag',               # High correlation with has_orientation
    'sdo2_orientation_Last_Event_Date_missing_flag',                # High correlation with has_orientation
    'gap_skc_to_start_weeks_missing_flag',                          # High correlation with has_skc
    'sdo2_skc_ADVIES_DEF_Niet_deelgenomen',                         # Is identical to has_skc feature
    'sdo2_orientation_Number_of_Events_Attended_missing_flag',      # Is identical to has_orientation feature
    'sdo2_orientation_Number_of_Event_Types_missing_flag',          # Is identical to attended_orientation_type_absent feature
    'attended_orientation_type_absent',                             # Is identical to sdo2_orientation_Number_of_Event_Types feature
    'sdo1_previous_Final_Exam_Date_missing_flag',                   # Is identical to has_previous feature
    'sdo1_previous_Previous_Education_Type_Unknown',                # Is almost identical to has_previous feature
    'sdo1_postal_distance_distance_to_3584_CS_missing_flag',        # High correlation with sdo5_degree_POSTAL_COUNTRY_NL feature
    'has_orientation',                                              # Is redundant since sdo2_orientation_Number_of_Events_Attended already encodes this information
    'time_first_orient_to_start_weeks_missing_flag',                # High correlation with sdo2_orientation_Number_of_Events_Attended
    'time_last_orient_to_start_weeks_missing_flag',                 # High correlation with sdo2_orientation_Number_of_Events_Attended
    'sdo6_results_Percentage_Credits_A',                            # Remove derived feature, since it might introduce multicollinearity with sdo6_results_Credits_Block_A and/or sdo6_results_Potential_Credits_A
    'sdo6_results_Percentage_Credits_A_missing_flag',               # Feature derived is removed above
    'SINH_ID',                                                      # Only needed for testing-purposes
    'Multiple_Degrees_Enrollment',
    'sdo2_skc_ADVIES_DEF_Others',
    # sdo78 missing flags: dropped because sdo78_response_type already captures
    # the full missingness structure (Non-responder / Partial-responder / Complete-responder)
    'sdo78_L51 (Study effort)_missing_flag',
    'sdo78_L50 (Alignment with prior education)_missing_flag',
    'sdo78_HU21_13 (General well-being)_missing_flag',
    'sdo78_L129 (Stress)_missing_flag',
    'sdo78_Support_Avg_missing_flag',
    'sdo78_Belonging_Avg_missing_flag',
    'sdo78_L110 (Motivation to complete the program)_missing_flag',
    'sdo78_L3 (Satisfaction with choide of degree)_missing_flag'
]
data = data.drop(columns=columns_to_drop, errors='ignore')

In [ ]:
# Select training data
data_train = data[data['set'] == 'train']
data_train

Define which columns to ignore (these are not considered features):

In [ ]:
columns_to_ignore = ['set']
data_train = data_train.drop(columns=columns_to_ignore, errors='ignore')

### 3.1 Variance Threshold Filtering

Removes features with little to no variation.

In [ ]:
# Investigate features with little to no variation
# For each numeric column, calculate variance

# Identify numeric columns (exclude string/object columns)
numeric_cols = data_train.select_dtypes(include=[np.number]).columns.tolist()

# Calculate variance for each numeric column
variance_analysis = pd.DataFrame({
    'column': numeric_cols,
    'variance': [data_train[col].var() for col in numeric_cols],
    'std_dev': [data_train[col].std() for col in numeric_cols],
    'unique_values': [data_train[col].nunique() for col in numeric_cols],
    'data_type': [str(data_train[col].dtype) for col in numeric_cols]
}).sort_values('variance')

print(f"📊 Variance Analysis for {len(numeric_cols)} numeric features:\n")

# Show features with lowest variance (potential candidates for removal)
print("🔻 Features with LOWEST variance (top 20):")
print(variance_analysis.head(20)[['column', 'variance', 'unique_values']].to_string(index=False))

print(f"\n🔺 Features with HIGHEST variance (top 10):")
print(variance_analysis.tail(10)[['column', 'variance', 'unique_values']].to_string(index=False))

# Identify zero-variance features
zero_variance = variance_analysis[variance_analysis['variance'] == 0.0]
print(f"\n⚠️  Features with ZERO variance: {len(zero_variance)}")
if len(zero_variance) > 0:
    print(zero_variance[['column', 'unique_values']].to_string(index=False))

# Identify very low variance features (< 0.01)
low_variance = variance_analysis[
    (variance_analysis['variance'] < 0.01) & (variance_analysis['variance'] > 0)
]
print(f"\n⚠️  Features with very LOW variance (< 0.01): {len(low_variance)}")
if len(low_variance) > 0:
    print(low_variance[['column', 'variance', 'unique_values']].to_string(index=False))

### 3.2 Feature correlation
Examines pairs of highly correlated feature pairs. 

In [ ]:
# Create hybrid correlation matrix: Pearson for binary-binary, Spearman for others
def create_hybrid_correlation_matrix(data, numeric_cols):
    """
    Create correlation matrix using:
    - Pearson correlation for binary-binary feature pairs
    - Spearman correlation for all other combinations
    """
    # Initialize with Spearman correlation as base
    correlation_matrix = data[numeric_cols].corr(method='spearman')
    
    # Identify binary columns (only values 0 and 1)
    binary_cols = []
    for col in numeric_cols:
        unique_vals = set(data[col].dropna().unique())
        if unique_vals.issubset({0, 1}):
            binary_cols.append(col)
    
    print(f"📊 Identified {len(binary_cols)} binary columns for Pearson correlation")
    
    # Calculate Pearson correlation for binary-binary pairs
    if len(binary_cols) > 1:
        pearson_matrix = data[binary_cols].corr(method='pearson')
        
        # Replace binary-binary correlations with Pearson values
        for i, col1 in enumerate(binary_cols):
            for j, col2 in enumerate(binary_cols):
                if i != j:  # Don't replace diagonal (self-correlation)
                    correlation_matrix.loc[col1, col2] = pearson_matrix.loc[col1, col2]
    
    return correlation_matrix

# Apply hybrid correlation method
correlation_matrix = create_hybrid_correlation_matrix(data_train, numeric_cols)

print("🔄 Hybrid correlation matrix created:")
print("   - Pearson correlation for binary-binary feature pairs")
print("   - Spearman correlation for all other combinations")

# Visualize correlation matrix (optional)
import seaborn as sns
import matplotlib.pyplot as plt 
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, cmap='coolwarm', center=0, square=True)
plt.title("Feature Correlation Matrix")
plt.show()

# Print highly correlated feature pairs
threshold = 0.9
high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i):
        if correlation_matrix.iloc[i, j] > threshold or correlation_matrix.iloc[i, j] < -threshold:
            high_corr_pairs.append((correlation_matrix.columns[i], correlation_matrix.columns[j], correlation_matrix.iloc[i, j]))

print(f"\n⚠️  Highly correlated feature pairs (correlation > {threshold}): {len(high_corr_pairs)}")
for pair in high_corr_pairs:
    print(f"{pair[0]} and {pair[1]}: correlation = {pair[2]:.2f}")

`Has_nf` and `has_skc` are expected to strongly correlate (negatively): all students enrolling in a numerus fixus degree (`has_nf = 1`) do not participate in the skc (`has_skc = 0`). The correlation is not perfect since there are many students that for other reasons have no skc data available, keeping both features is therefore advised. 

Delete 'has_previous' and keep sdo1_previous_Multiple_Previous_Education_missing_flag because its more explicit 

In [ ]:
del data['sdo1_previous_Multiple_Previous_Education_missing_flag']

#### 3.2.1 Correlation with outcome (predictor) feature

In [ ]:
# Look at corrections with target variable sdo5_degree_drop_out
target_variable = 'sdo5_degree_drop_out'

target_correlations = correlation_matrix[target_variable].sort_values(ascending=False)
print(f"\n🔍 Correlation of features with target variable '{target_variable}' (desc):")
print(target_correlations.to_string())

### Save selected features to CSV

In [ ]:
# Save dataset
output_path = OUT_DIR / "feature_selection_all.csv"
data.to_csv(output_path, index=False)

print("✅ Feature selection complete!")
print(f"   📂 Saved: {output_path}")
print(f"   📊 Shape: {data.shape}")
print(f"   🏷️ Columns: {len(data.columns)}")

# Show data type summary
print(f"\n📋 Data type summary:")
dtype_counts = data.dtypes.value_counts()
for dtype, count in dtype_counts.items():
    print(f"   {dtype}: {count} columns")

## 3.3. Permutation importance {#permutation-importance}

Global / model-agnostic feature selection

This script computes cross-validated permutation feature
importance and uses it as a global feature selection
technique that is largely model-agnostic. The goal is not to
optimize features for a specific model via iterative subset
selection (e.g., RFE), but to identify features that show
consistent predictive value on unseen data and can therefore
be shared across different model families.

Why permutation importance (ROC-AUC based):

Permutation importance measures the drop in ROC-AUC when a
feature column is randomly shuffled, breaking its
relationship with the target while keeping all other
features unchanged. A larger drop indicates that the model
relied on that feature to separate the classes.

Using ROC-AUC provides a threshold-independent and
class-balanced measure of feature contribution, making it
suitable for imbalanced classification and global feature
assessment.

Evaluating importance on held-out validation folds helps
identify features that contribute to generalization, not
just training fit.

Role of stability (robustness across folds):

Stability measures how consistently a feature contributes
positively across cross-validation folds, complementing
average permutation importance with a robustness criterion.

With 5-fold cross-validation:
stability = 1.0 -> helpful in all 5 folds
stability = 0.8 -> helpful in 4 out of 5 folds
stability = 0.6 -> helpful in 3 out of 5 folds
stability = 0.2 -> helpful in only 1 fold
stability = 0.0 -> never helpful

This interpretation makes stability an intuitive measure
of reproducibility and guards against fold-specific noise,
rare-category effects, and spurious correlations.

Feature elimination:

Features are marked for removal only when they show no
average contribution (non-positive mean importance) and
low stability (at most one fold) across folds, ensuring that only consistently
non-informative features are dropped while avoiding
over-pruning.

#### 3.3.1 Permutation importance original method

In [ ]:
# ============================================================
# Scenario 1: Original method
# Cross-validated permutation importance + stability
# ============================================================

# ------------------------------------------------------------
# Prepare target and features (TRAINING DATA ONLY)
# ------------------------------------------------------------
X_original = data_train.drop("sdo5_degree_drop_out", axis=1).copy()
y_original = data_train["sdo5_degree_drop_out"].copy()

# Drop remaining object columns (e.g., dates stored as strings)
object_cols_original = X_original.select_dtypes(include=["object"]).columns
X_original = X_original.drop(columns=object_cols_original)

print(f"Training data shape: {X_original.shape}")
print(f"Target distribution:\n{y_original.value_counts()}")

# ------------------------------------------------------------
# Random Forest used only as an importance estimator
# ------------------------------------------------------------
rf_model_original = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

# ------------------------------------------------------------
# Cross-validated permutation importance (evaluated on CV folds)
# ------------------------------------------------------------
print("\n🔁 Calculating CV-based permutation importance...")

cv_original = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_importances_original = []
for fold_original, (train_idx_original, val_idx_original) in enumerate(
    cv_original.split(X_original, y_original), start=1
):
    X_tr_original = X_original.iloc[train_idx_original]
    X_val_original = X_original.iloc[val_idx_original]
    y_tr_original = y_original.iloc[train_idx_original]
    y_val_original = y_original.iloc[val_idx_original]

    rf_model_original.fit(X_tr_original, y_tr_original)

    pi_original = permutation_importance(
        rf_model_original,
        X_val_original,
        y_val_original,
        n_repeats=10,
        random_state=42,
        n_jobs=-1,
        scoring="roc_auc"
    )

    fold_importances_original.append(pi_original.importances_mean)
    print(f"   ✅ Fold {fold_original} done")

fold_importances_original = np.vstack(fold_importances_original)

# ------------------------------------------------------------
# Aggregate importance across folds
# ------------------------------------------------------------
importance_mean_cv_original = fold_importances_original.mean(axis=0)
importance_std_cv_original = fold_importances_original.std(axis=0)
stability_original = (fold_importances_original > 0).mean(axis=0)

perm_importance_df_original = (
    pd.DataFrame({
        "feature": X_original.columns,
        "importance_mean": importance_mean_cv_original,
        "importance_std": importance_std_cv_original,
        "stability": stability_original
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Inspect results
# ------------------------------------------------------------
print("\n📊 Top 20 most important features (Original CV Permutation Importance):")
print(perm_importance_df_original.head(20).to_string(index=False))

print("\n📊 Bottom 20 least important features:")
print(perm_importance_df_original.tail(20).to_string(index=False))

# ------------------------------------------------------------
# Visualization
# ------------------------------------------------------------
plt.figure(figsize=(10, 8))
top_n_original = 20
top_features_original = perm_importance_df_original.head(top_n_original).iloc[::-1]

plt.barh(top_features_original["feature"], top_features_original["importance_mean"])
plt.xlabel("Mean CV Permutation Importance (ROC-AUC)")
plt.title(f"Top {top_n_original} Features by Original CV Permutation Importance")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Conservative feature dropping
# ------------------------------------------------------------
drop_features_original = perm_importance_df_original[
    (perm_importance_df_original["importance_mean"] <= 0) &
    (perm_importance_df_original["stability"] <= 0.2)
]["feature"].tolist()

print(f"\nSuggested drop candidates (original): {len(drop_features_original)}")
if drop_features_original:
    for f_original in drop_features_original:
        print(f"  - {f_original}")
else:
    print("  None")

#### 3.3.2 Permutation importance optimized method

In [ ]:
# NOTE:

# A) Cross-validation robustness improvement
# The original implementation included 5-fold cross-validation and
# 10 permutation repeats within each validation fold. However, it did
# not include repeated cross-validation across multiple resampled split
# configurations, so stability was assessed only across folds from one
# single CV partitioning of the data.

# Original code:
# - 5 folds
# - each fold uses 10 permutation shuffles
# - total importance estimates per feature: 5 fold-level means

# New code:
# - 5 folds × 5 repeated CV rounds = 25 folds
# - each fold still uses 10 permutation shuffles
# - total importance estimates per feature: 25 fold-level means

# Improvement:
# - the original approach stabilizes shuffle noise (within-fold randomness)
# - the updated approach also stabilizes data split variability across folds


# B) Uncertainty-aware feature elimination
# The original feature elimination rule relied on a strict zero threshold
# (importance_mean <= 0), without incorporating the standard deviation.

# Improvement:
# - standard deviation is now used as an estimate of uncertainty
# - features are evaluated based on whether their importance exceeds noise

# Conceptual shift:
# - from: “is importance > 0 in this run?”
# - to:   “is importance consistently meaningful across many runs?”

In [ ]:
# ============================================================
# Scenario 2: Optimized method
# Repeated cross-validated permutation importance + stability
# ============================================================

# ------------------------------------------------------------
# Prepare target and features (TRAINING DATA ONLY)
# ------------------------------------------------------------
X = data_train.drop("sdo5_degree_drop_out", axis=1).copy()
y = data_train["sdo5_degree_drop_out"].copy()

# Drop remaining object columns (e.g., dates stored as strings)
object_cols = X.select_dtypes(include=["object"]).columns
X = X.drop(columns=object_cols)

print(f"Training data shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")

# ------------------------------------------------------------
# Random Forest used only as an importance estimator
# ------------------------------------------------------------
rf_params = {
    "n_estimators": 100,
    "max_depth": 10,
    "n_jobs": -1,
    "class_weight": "balanced"
}

# ------------------------------------------------------------
# Repeated cross-validated permutation importance
# ------------------------------------------------------------
print("\n🔁 Calculating repeated CV-based permutation importance...")

n_splits = 5
n_repeats_cv = 5
perm_repeats = 10
base_seed = 42

cv = RepeatedStratifiedKFold(
    n_splits=n_splits,
    n_repeats=n_repeats_cv,
    random_state=base_seed
)

all_importances = []
fold_meta = []

for split_idx, (train_idx, val_idx) in enumerate(cv.split(X, y), start=1):
    X_tr = X.iloc[train_idx]
    X_val = X.iloc[val_idx]
    y_tr = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    rf_model = RandomForestClassifier(
        **rf_params,
        random_state=base_seed + split_idx
    )

    rf_model.fit(X_tr, y_tr)

    pi = permutation_importance(
        rf_model,
        X_val,
        y_val,
        n_repeats=perm_repeats,
        random_state=base_seed + split_idx,
        n_jobs=-1,
        scoring="roc_auc"
    )

    all_importances.append(pi.importances_mean)
    fold_meta.append(split_idx)

    print(f"   ✅ Split {split_idx}/{n_splits * n_repeats_cv} done")

all_importances = np.vstack(all_importances)

# ------------------------------------------------------------
# Aggregate importance across all folds and repeats
# ------------------------------------------------------------
importance_mean_cv = all_importances.mean(axis=0)
importance_std_cv = all_importances.std(axis=0)
stability = (all_importances > 0).mean(axis=0)

perm_importance_df = (
    pd.DataFrame({
        "feature": X.columns,
        "importance_mean": importance_mean_cv,
        "importance_std": importance_std_cv,
        "stability": stability
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

perm_importance_df["mean_lt_std"] = (
    perm_importance_df["importance_mean"] < perm_importance_df["importance_std"]
)

# ------------------------------------------------------------
# Inspect results
# ------------------------------------------------------------
print("\n📊 Top 20 most important features (Repeated CV Permutation Importance):")
print(perm_importance_df.head(20).to_string(index=False))

print("\n📊 Bottom 20 least important features:")
print(perm_importance_df.tail(20).to_string(index=False))

# ------------------------------------------------------------
# Visualization
# ------------------------------------------------------------
plt.figure(figsize=(10, 8))
top_n = 20
top_features = perm_importance_df.head(top_n).iloc[::-1]

plt.barh(top_features["feature"], top_features["importance_mean"])
plt.xlabel("Mean Repeated-CV Permutation Importance (ROC-AUC)")
plt.title(f"Top {top_n} Features by Repeated-CV Permutation Importance")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Conservative feature dropping
# ------------------------------------------------------------
stability_threshold = 0.2

drop_features = perm_importance_df[
    (perm_importance_df["importance_mean"] < perm_importance_df["importance_std"]) &
    (perm_importance_df["stability"] <= stability_threshold)
]["feature"].tolist()

print(f"\nSuggested drop candidates (optimized): {len(drop_features)}")
if drop_features:
    for f in drop_features:
        print(f"  - {f}")
else:
    print("  None")

# ------------------------------------------------------------
# Optional: keep a cleaned training set
# ------------------------------------------------------------
X_selected = X.drop(columns=drop_features, errors="ignore")
print(f"\nSelected feature set shape: {X_selected.shape}")

#### 3.3.3 Permutation importance optimized method without sd078 data

In [ ]:
# Create a copy of the data without columns starting with 'sdo78'
data_no_sdo78 = data.loc[:, ~data.columns.str.startswith("sdo78")].copy()

print(f"Original shape: {data.shape}")
print(f"Filtered shape (without sdo78): {data_no_sdo78.shape}")

# Optional: check removed columns
removed_cols = [col for col in data.columns if col.startswith("sdo78")]
print(f"\nRemoved columns ({len(removed_cols)}):")
for col in removed_cols:
    print(f"  - {col}")

In [ ]:
# ============================================================
# Scenario 3: Optimized method without sdo78 columns
# Repeated cross-validated permutation importance + stability
# ============================================================

# ------------------------------------------------------------
# Prepare filtered TRAINING dataset (remove sdo78 columns)
# ------------------------------------------------------------
data_train_no_sdo78 = data_train.loc[:, ~data_train.columns.str.startswith("sdo78")].copy()

X_no_sdo78 = data_train_no_sdo78.drop("sdo5_degree_drop_out", axis=1).copy()
y_no_sdo78 = data_train_no_sdo78["sdo5_degree_drop_out"].copy()

# Drop remaining object columns
object_cols_no_sdo78 = X_no_sdo78.select_dtypes(include=["object"]).columns
X_no_sdo78 = X_no_sdo78.drop(columns=object_cols_no_sdo78)

print(f"Filtered training data shape: {X_no_sdo78.shape}")
print(f"Target distribution:\n{y_no_sdo78.value_counts()}")

# ------------------------------------------------------------
# Random Forest used only as an importance estimator
# ------------------------------------------------------------
rf_params_no_sdo78 = {
    "n_estimators": 100,
    "max_depth": 10,
    "n_jobs": -1,
    "class_weight": "balanced"
}

# ------------------------------------------------------------
# Repeated cross-validated permutation importance
# ------------------------------------------------------------
print("\n🔁 Calculating repeated CV-based permutation importance (without sdo78)...")

n_splits_no_sdo78 = 5
n_repeats_cv_no_sdo78 = 5
perm_repeats_no_sdo78 = 10
base_seed_no_sdo78 = 42

cv_no_sdo78 = RepeatedStratifiedKFold(
    n_splits=n_splits_no_sdo78,
    n_repeats=n_repeats_cv_no_sdo78,
    random_state=base_seed_no_sdo78
)

all_importances_no_sdo78 = []

for split_idx_no_sdo78, (train_idx_no_sdo78, val_idx_no_sdo78) in enumerate(
    cv_no_sdo78.split(X_no_sdo78, y_no_sdo78), start=1
):
    X_tr_no_sdo78 = X_no_sdo78.iloc[train_idx_no_sdo78]
    X_val_no_sdo78 = X_no_sdo78.iloc[val_idx_no_sdo78]
    y_tr_no_sdo78 = y_no_sdo78.iloc[train_idx_no_sdo78]
    y_val_no_sdo78 = y_no_sdo78.iloc[val_idx_no_sdo78]

    rf_model_no_sdo78 = RandomForestClassifier(
        **rf_params_no_sdo78,
        random_state=base_seed_no_sdo78 + split_idx_no_sdo78
    )

    rf_model_no_sdo78.fit(X_tr_no_sdo78, y_tr_no_sdo78)

    pi_no_sdo78 = permutation_importance(
        rf_model_no_sdo78,
        X_val_no_sdo78,
        y_val_no_sdo78,
        n_repeats=perm_repeats_no_sdo78,
        random_state=base_seed_no_sdo78 + split_idx_no_sdo78,
        n_jobs=-1,
        scoring="roc_auc"
    )

    all_importances_no_sdo78.append(pi_no_sdo78.importances_mean)
    print(
        f"   ✅ Split {split_idx_no_sdo78}/"
        f"{n_splits_no_sdo78 * n_repeats_cv_no_sdo78} done"
    )

all_importances_no_sdo78 = np.vstack(all_importances_no_sdo78)

# ------------------------------------------------------------
# Aggregate importance across all folds and repeats
# ------------------------------------------------------------
importance_mean_no_sdo78 = all_importances_no_sdo78.mean(axis=0)
importance_std_no_sdo78 = all_importances_no_sdo78.std(axis=0)
stability_no_sdo78 = (all_importances_no_sdo78 > 0).mean(axis=0)

perm_importance_df_no_sdo78 = (
    pd.DataFrame({
        "feature": X_no_sdo78.columns,
        "importance_mean": importance_mean_no_sdo78,
        "importance_std": importance_std_no_sdo78,
        "stability": stability_no_sdo78
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

perm_importance_df_no_sdo78["mean_lt_std"] = (
    perm_importance_df_no_sdo78["importance_mean"] <
    perm_importance_df_no_sdo78["importance_std"]
)

# ------------------------------------------------------------
# Inspect results
# ------------------------------------------------------------
print("\n📊 Top 20 most important features (without sdo78):")
print(perm_importance_df_no_sdo78.head(20).to_string(index=False))

print("\n📊 Bottom 20 least important features (without sdo78):")
print(perm_importance_df_no_sdo78.tail(20).to_string(index=False))

# ------------------------------------------------------------
# Visualization
# ------------------------------------------------------------
plt.figure(figsize=(10, 8))
top_n_no_sdo78 = 20
top_features_no_sdo78 = perm_importance_df_no_sdo78.head(top_n_no_sdo78).iloc[::-1]

plt.barh(top_features_no_sdo78["feature"], top_features_no_sdo78["importance_mean"])
plt.xlabel("Mean Repeated-CV Permutation Importance (ROC-AUC)")
plt.title(f"Top {top_n_no_sdo78} Features by Repeated-CV Permutation Importance (without sdo78)")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Conservative feature dropping for stability test only
# ------------------------------------------------------------
stability_threshold_no_sdo78 = 0.2

drop_features_no_sdo78 = perm_importance_df_no_sdo78[
    (perm_importance_df_no_sdo78["importance_mean"] < perm_importance_df_no_sdo78["importance_std"]) &
    (perm_importance_df_no_sdo78["stability"] <= stability_threshold_no_sdo78)
]["feature"].tolist()

print(f"\nSuggested drop candidates for stability test only: {len(drop_features_no_sdo78)}")
if drop_features_no_sdo78:
    for f_no_sdo78 in drop_features_no_sdo78:
        print(f"  - {f_no_sdo78}")
else:
    print("  None")

X_selected_no_sdo78 = X_no_sdo78.drop(columns=drop_features_no_sdo78, errors="ignore")
print(f"\nSelected feature set shape (without sdo78): {X_selected_no_sdo78.shape}")

## 5. Final Feature Set {#final-feature-set}

Select the optimal feature subset and prepare the final dataset for model training.

In [ ]:
# ============================================================
# Apply global feature selection: drop selected features
# (using drop_features computed from CV permutation importance)
# ============================================================

print(f"🔻 Dropping {len(drop_features)} features from data:")
for f in drop_features:
    print(f"  - {f}")

# Drop features from data only
data = data.drop(columns=drop_features, errors="ignore")

print("\n✅ Feature drop complete")
print(f"   New data shape: {data.shape}")

In [ ]:
# Save dataset
output_path = OUT_DIR / "feature_selection.csv"
data.to_csv(output_path, index=False)

print("✅ Feature selection complete!")
print(f"   📂 Saved: {output_path}")
print(f"   📊 Shape: {data.shape}")
print(f"   🏷️ Columns: {len(data.columns)}")

# Show data type summary
print(f"\n📋 Data type summary:")
dtype_counts = data.dtypes.value_counts()
for dtype, count in dtype_counts.items():
    print(f"   {dtype}: {count} columns")